# ML Validation

In [ ]:
pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   ----------

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [6]:
df = pd.read_csv('f1_clean_data.csv')
df = df.dropna(subset=['Finishing_Tier'])
df

,Race_ID,Driver,LapNumber,Finishing_Tier,Tire_Age,Fuel_Mass_Proxy,Degradation_x_Fuel,Tire_MEDIUM,Tire_SOFT,Tire_Age.1
0,2023_Bahrain,VER,1.0,Podium,4.0,108.070175,-301.785428,0,1,4.0
1,2023_Bahrain,VER,2.0,Podium,5.0,106.140351,-240.583466,0,1,5.0
2,2023_Bahrain,VER,3.0,Podium,6.0,104.210526,-183.241153,0,1,6.0
3,2023_Bahrain,VER,4.0,Podium,7.0,102.280702,-129.758489,0,1,7.0
4,2023_Bahrain,VER,5.0,Podium,8.0,100.350877,-80.135474,0,1,8.0
...,...,...,...,...,...,...,...,...,...,...
1050,2023_Bahrain,PIA,9.0,No Points,9.0,92.631579,-29.663702,0,1,9.0
1051,2023_Bahrain,PIA,10.0,No Points,10.0,90.701754,6.450541,0,1,10.0
1052,2023_Bahrain,PIA,11.0,No Points,11.0,88.771930,38.705135,0,1,11.0
1053,2023_Bahrain,PIA,12.0,No Points,12.0,86.842105,67.100079,0,1,12.0


In [7]:
#features and targets
base_features = ['Tire_Age', 'Fuel_Mass_Proxy'] + [col for col in df.columns if col.startswith('Tire_') and col not in ['Tire_Age', 'Tire_Age.1']]

X = df[base_features]
y = df['Finishing_Tier']
groups = df['Race_ID']

In [ ]:
model = LogisticRegression(multi_class='multinomial', solver='saga', max_iter=1000)

#Leave-One-Group-Out Cross-Validation (LOGO-CV)
logo = LeaveOneGroupOut()

all_y_true = []
all_y_pred = []

print("LOGO-CV")


for train_index, test_index in logo.split(X, y, groups):
    # Split the data
    X_train, X_test = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    

    train_mean_tire = X_train['Tire_Age'].mean()
    train_mean_fuel = X_train['Fuel_Mass_Proxy'].mean()
    
    X_train['Degradation_x_Fuel'] = (X_train['Tire_Age'] - train_mean_tire) * (X_train['Fuel_Mass_Proxy'] - train_mean_fuel)
    X_test['Degradation_x_Fuel'] = (X_test['Tire_Age'] - train_mean_tire) * (X_test['Fuel_Mass_Proxy'] - train_mean_fuel)
    

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    

    all_y_true.extend(y_test)
    all_y_pred.extend(predictions)

#Assess Overall Fit (Classification Hit Ratio)
print("\nCLASSIFICATION HIT RATIO (UNSEEN RACES): ")
accuracy = accuracy_score(all_y_true, all_y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

print("CLASSIFICATION REPORT: ")
print(classification_report(all_y_true, all_y_pred))